# Encounter Map

Search for the current user's most recent encounters and plot their
locations on an interactive map.

**Extra dependency** — install before running this notebook:

```bash
uv pip install ipyleaflet
```

Set the usual environment variables (`WILDBOOK_URL`, `WILDBOOK_USERNAME`,
`WILDBOOK_PASSWORD`) before starting the kernel, or pass credentials
explicitly to `client.login()`.

In [37]:
import os
import getpass

from dotenv import load_dotenv
from pywildbook import WildbookClient
from ipyleaflet import Map, Marker, Popup, MarkerCluster
from ipywidgets import HTML

load_dotenv()

if not os.environ.get("WILDBOOK_URL"):
    os.environ["WILDBOOK_URL"] = input("WILDBOOK_URL: ")
if not os.environ.get("WILDBOOK_USERNAME"):
    os.environ["WILDBOOK_USERNAME"] = input("WILDBOOK_USERNAME: ")
if not os.environ.get("WILDBOOK_PASSWORD"):
    os.environ["WILDBOOK_PASSWORD"] = getpass.getpass("WILDBOOK_PASSWORD: ")

In [38]:
client = WildbookClient()
user = client.login()
print(f"Logged in as {user['username']}")

Logged in as kirk


In [39]:
results = client.search_encounters(
    client.filter_current_user(),
    size=50,
    sort='date',
    sort_order='desc'
)

encounters = results.get('hits', [])
print(f"Found {len(encounters)} encounters")

Found 10 encounters


In [40]:
def _popup(enc):
    """Return a Popup populated with encounter details."""
    genus = enc.get('genus', '')
    species = enc.get('specificEpithet', '')
    name = f"{genus} {species}".strip() or 'Unknown species'
    return Popup(
        children=[HTML(
            value=f"<b>{name}</b><br>"
            f"Year: {enc.get('year', 'N/A')}<br>"
            f"Location: {enc.get('verbatimLocality', 'N/A')}<br>"
            f"<small>ID: {enc.get('id', '')}</small>"
        )],
        max_width=250
    )


# Keep only encounters that carry a geo point
mapped = [
    e for e in encounters
    if isinstance(e.get('locationGeoPoint'), dict)
    and 'lat' in e['locationGeoPoint']
    and 'lon' in e['locationGeoPoint']
]
print(f"{len(mapped)} of {len(encounters)} encounters have a location")

if mapped:
    lats = [e['locationGeoPoint']['lat'] for e in mapped]
    lons = [e['locationGeoPoint']['lon'] for e in mapped]
    m = Map()
    markers = [
        Marker(
            location=[e['locationGeoPoint']['lat'], e['locationGeoPoint']['lon']],
            popup=_popup(e)
        )
        for e in mapped
    ]
    m.add_layer(MarkerCluster(markers=markers))
    # Fit the map to the bounding box of all plotted points
    m.fit_bounds([[min(lats), min(lons)], [max(lats), max(lons)]])
else:
    m = Map(center=[0, 0], zoom=2)

m

8 of 10 encounters have a location


Map(center=[0.0, 0.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_t…

In [41]:
client.logout()

True